# Math Basics for ML

In this notebook I demonstrate the math concepts behind the tools I used in the project. Instead of just calling library functions, I implement the calculations by hand using NumPy to show I understand what is happening underneath.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from numpy.linalg import norm

df = pd.read_csv("data/cleaned/titanic_cleaned.csv")
print("Loaded:", df.shape)

---

## Task 1 — Mean and Standard Deviation By Hand

**Mean** is the average — add up all values and divide by how many there are.

**Standard deviation** measures how spread out the values are around the mean. A high std means values vary a lot. A low std means they are close together.

The formula for std is:

`std = sqrt( sum of (each value - mean)^2 / total count )`

I calculate both manually using only NumPy array operations, then verify by comparing with NumPy's built-in `.mean()` and `.std()`.

In [ ]:
fare = df["Fare"].values

# Manual mean: sum all values, divide by count
manual_mean = np.sum(fare) / len(fare)

# Manual std: square root of average squared distance from mean
manual_std = np.sqrt(np.sum((fare - manual_mean) ** 2) / len(fare))

print(f"Manual  mean: {manual_mean:.2f}")
print(f"NumPy   mean: {np.mean(fare):.2f}")
print()
print(f"Manual  std:  {manual_std:.2f}")
print(f"NumPy   std:  {np.std(fare):.2f}")

Both results match exactly. The average fare paid by Titanic passengers was about **£31.22**, but the standard deviation of **£42.50** is actually larger than the mean — this tells me the fares were very spread out. Some people paid almost nothing while others paid hundreds of pounds.

---

## Task 2 — Standardisation By Hand (Broadcasting)

Standardisation (also called z-score normalisation) rescales a column so it has:
- **Mean = 0**
- **Standard deviation = 1**

The formula is: `z = (value - mean) / std`

This is called **broadcasting** in NumPy — instead of looping through each value manually, NumPy applies the formula to the entire array at once.

Why is this useful? Because different columns have different scales. Age goes from 0–80, Fare goes from 0–512. Standardising puts them on the same scale so one column doesn't dominate just because its numbers are bigger.

In [ ]:
# Standardise by hand using broadcasting
z_manual = (fare - manual_mean) / manual_std

# Standardise using sklearn for comparison
z_sklearn = StandardScaler().fit_transform(fare.reshape(-1, 1)).flatten()

print("Manual  z[:5]:", z_manual[:5].round(3))
print("Sklearn z[:5]:", z_sklearn[:5].round(3))
print()
print(f"Manual  result — mean: {z_manual.mean():.6f}, std: {z_manual.std():.4f}")
print(f"Sklearn result — mean: {z_sklearn.mean():.6f}, std: {z_sklearn.std():.4f}")

The manual result and sklearn's result are identical. After standardisation, the mean is essentially 0 (the tiny number like `2e-16` is just floating point rounding, not a real difference) and the std is 1. This confirms the formula works correctly.

I used this same technique in Phase 2 when I scaled the Age and Fare columns with StandardScaler.

---

## Task 3 — Cosine Similarity

Cosine similarity measures how **similar the direction** of two data points is, regardless of their size.

The formula is: `cos_sim = dot(A, B) / (norm(A) * norm(B))`

- **dot product** = multiply each pair of values and sum them up
- **norm** = the length (size) of the vector

The result is always between -1 and 1:
- **1** = identical direction (very similar)
- **0** = no relationship
- **-1** = completely opposite

I compare the passenger who paid the **highest fare** versus the passenger who paid the **lowest fare** to see how different they are across all numeric features.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

highest = df.loc[df["Fare"].idxmax(), numeric_cols].values.astype(float)
lowest  = df.loc[df["Fare"].idxmin(), numeric_cols].values.astype(float)

# Dot product
dot_product = np.dot(highest, lowest)

# Norms (lengths of each vector)
norm_highest = norm(highest)
norm_lowest  = norm(lowest)

# Cosine similarity
cos_sim = dot_product / (norm_highest * norm_lowest)

print(f"Highest fare passenger: {df.loc[df['Fare'].idxmax(), 'Fare']:.2f}")
print(f"Lowest  fare passenger: {df.loc[df['Fare'].idxmin(), 'Fare']:.2f}")
print()
print(f"Dot product:     {dot_product:.2f}")
print(f"Norm (highest):  {norm_highest:.2f}")
print(f"Norm (lowest):   {norm_lowest:.2f}")
print(f"Cosine similarity: {cos_sim:.4f}")

The cosine similarity is **0.1241** — close to 0, meaning these two passengers are quite different from each other across all their features. This makes sense: the highest-fare passenger was likely a 1st class traveller with a large family, while the lowest-fare passenger was probably a solo 3rd class traveller. Despite both being on the same ship, their profiles are almost completely different.

---

## Task 4 — Probability Estimate

Probability is the chance that something happens, expressed as a number between 0 and 1 (or 0% to 100%).

The formula for empirical probability (calculated from real data) is:

`P(event) = number of times event happened / total number of cases`

I calculate the probability of survival for each passenger class — this is called a **conditional probability**: given that someone was in a specific class, what was the probability they survived?

In [ ]:
# P(Survived | each class)
for pclass in [1, 2, 3]:
    group = df[df["Pclass"] == pclass]
    prob = group["Survived"].sum() / len(group)
    print(f"P(Survived | {pclass}st/nd/rd class) = {prob:.2%}  ({int(group['Survived'].sum())} survived out of {len(group)})")

print()

# P(Survived | Sex)
for sex in ["female", "male"]:
    group = df[df["Sex"] == sex]
    prob = group["Survived"].sum() / len(group)
    print(f"P(Survived | {sex}) = {prob:.2%}  ({int(group['Survived'].sum())} survived out of {len(group)})")

These numbers confirm everything I found in the EDA:

- A **1st class passenger** had a ~63% chance of surviving
- A **3rd class passenger** had only a ~24% chance — less than half of 1st class
- A **female passenger** had a ~74% chance of surviving
- A **male passenger** had only a ~19% chance

These are empirical probabilities — calculated directly from the data, not assumed. They tell a clear story: survival on the Titanic was strongly linked to wealth and gender, not to luck.

---

## Summary

In this notebook I demonstrated 4 core math concepts used in data science and machine learning:

| Task | Formula | Result |
|------|---------|--------|
| Mean | sum(x) / n | Fare mean = 31.22 |
| Std deviation | sqrt(sum((x - mean)^2) / n) | Fare std = 42.50 |
| Standardisation | z = (x - mean) / std | Mean → 0, Std → 1 |
| Cosine similarity | dot(A,B) / (norm(A) * norm(B)) | 0.1241 (very different passengers) |
| Probability | count(event) / total | 63% survival for 1st class |

All manual calculations matched the library outputs exactly, confirming the formulas are correct.